In [2]:
# SO CONFIGURANDO AMBIENTE

# Instalação do Pyomo e do Solver GLPK (Comando para Google Colab ou VS Code Terminal)
!pip install pyomo

import pyomo.environ as pyo
from pyomo.opt import SolverFactory

# Criamos o objeto do solver que será usado mais adiante
opt = SolverFactory('glpk', executable='/opt/homebrew/bin/glpsol')

print("Ambiente configurado com sucesso!")

Ambiente configurado com sucesso!


In [3]:

# 1. Entrada de Dados
origens = ['Uberlandia', 'Rio_Verde']
destinos = ['Santos', 'Paranagua']

oferta = {'Uberlandia': 300, 'Rio_Verde': 500}
demanda = {'Santos': 400, 'Paranagua': 400}

custos = {
    ('Uberlandia', 'Santos'): 10, ('Uberlandia', 'Paranagua'): 15,
    ('Rio_Verde', 'Santos'): 12, ('Rio_Verde', 'Paranagua'): 11
}

# 2. Criação do Modelo
model = pyo.ConcreteModel()

# 3. Variáveis de Decisão: x[i,j] quantidade enviada de i para j
model.x = pyo.Var(origens, destinos, within=pyo.NonNegativeReals)

# 4. Função Objetivo: Minimização dos custos
model.obj = pyo.Objective(expr=sum(model.x[i,j] * custos[i,j] for i in origens for j in destinos), sense=pyo.minimize)

# 5. Restrições de Oferta
# O sum() percorre todas os destinos 'j' para a origem fixo 'i'
# Retorna para o Pyomo a relação de desigualdade. O solver verá algo como: (xi,1 + xi,2+...+xi,n ≤ oferta[i]).

def rest_oferta(model, i):
    return sum(model.x[i,j] for j in destinos) == oferta[i]
model.rest_oferta = pyo.Constraint(origens, rule=rest_oferta)

# 6. Restrições de Demanda
# O sum() percorre todas as origens 'i' para o destino fixo 'j'
# Retorna para o Pyomo a relação de desigualdade. O solver verá algo como: (x1,j + x2,j+...+ xn,j ≥ demanda[j]).

def rest_demanda(model, j):
    return sum(model.x[i,j] for i in origens) == demanda[j]
model.rest_demanda = pyo.Constraint(destinos, rule=rest_demanda)

# 7. ATIVAÇÃO DA SENSIBILIDADE (IMPORTANTE)
# O sufixo 'dual' permite acessar os Preços Sombra e Custos Reduzidos
model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

# 8. Solução
opt = pyo.SolverFactory('glpk', executable='/opt/homebrew/bin/glpsol')
results = opt.solve(model)

print(f"\nStatus da Solução: {results.solver.status}")
print(f"Valor Ótimo: {pyo.value(model.obj):.2f}")

# 9 .Exibir resultados
for i in origens:
    for j in destinos:
        if pyo.value(model.x[i,j]) > 0:
            print(f"Enviar {pyo.value(model.x[i,j])} de {i} para {j}")

# 10. Gerando o relatório de sensibilidade completo em arquivo texto

# O sufixo 'ranges' cria o relatório de sensibilidade textual do GLPK

opt.solve(model, options={'ranges': 'relatorio_sensibilidade.txt'})

# Lendo os intervalos de estabilidade do arquivo gerado

with open('relatorio_sensibilidade.txt', 'r') as file:
    # Mostra as linhas que contêm os limites de estabilidade (Upper e Lower bounds)
    print(file.read())


Status da Solução: ok
Valor Ótimo: 8600.00
Enviar 300.0 de Uberlandia para Santos
Enviar 100.0 de Rio_Verde para Santos
Enviar 400.0 de Rio_Verde para Paranagua
GLPK 5.0  - SENSITIVITY ANALYSIS REPORT                                                                         Page   1

Problem:    
Objective:  x1 = 8600 (MINimum)

   No. Row name     St      Activity         Slack   Lower bound       Activity      Obj coef  Obj value at Limiting
                                          Marginal   Upper bound          range         range   break point variable
------ ------------ -- ------------- ------------- -------------  ------------- ------------- ------------- ------------
     1 c_e_x6_      NS     300.00000        .          300.00000      300.00000          -Inf    8600.00000 c_e_x9_
                                           9.00000     300.00000      300.00000          +Inf    8600.00000 c_e_x9_

     2 c_e_x7_      NS     500.00000        .          500.00000      500.00000   

--------------------------------------------------


AGORA UM PROBLEMA DE TRANSBORDO

In [4]:
import pyomo.environ as pyo

# 1. Definição dos Dados de Entrada
nos = ['P1', 'P2', 'T1', 'T2', 'D1', 'D2', 'D3']

# Ofertas e Demandas Originais
oferta_orig = {'P1': 1000, 'P2': 1200, 'T1': 0, 'T2': 0, 'D1': 0, 'D2': 0, 'D3': 0}
demanda_orig = {'P1': 0, 'P2': 0, 'T1': 0, 'T2': 0, 'D1': 800, 'D2': 900, 'D3': 500}

# Cálculo do Buffer (Quantidade Tampão)
Oftotal = sum(oferta_orig.values()) # Oftotal = 2200
Dtotal = sum(demanda_orig.values()) # Dtotal = 2200

if Oftotal >= Dtotal:
    B = Oftotal
else:
    B = Dtotal


# Ajuste para o Modelo de Transporte (Metodologia Tampão)
# Nós de transbordo e destino recebem B na oferta e demanda para permitir fluxo passante
oferta_ajustada = {i: oferta_orig[i] + (B if i in ['T1', 'T2', 'D1', 'D2'] else 0) for i in nos}
demanda_ajustada = {j: demanda_orig[j] + (B if j in ['T1', 'T2', 'D1', 'D2'] else 0) for j in nos}

# Custos das rotas indicadas nos arcos da image_782b93.jpg
custos = {
    ('P1', 'T1'): 3, ('P1', 'T2'): 4,
    ('P2', 'T1'): 2, ('P2', 'T2'): 5,
    ('T1', 'D1'): 8, ('T1', 'D2'): 6, ('T1', 'T2'): 7,
    ('T2', 'D2'): 4, ('T2', 'D3'): 9,
    ('D1', 'D2'): 5,
    ('D2', 'D3'): 3,
    # Auto-arcos com custo zero para a quantidade tampão não utilizada
    ('T1', 'T1'): 0, ('T2', 'T2'): 0, ('D1', 'D1'): 0, ('D2', 'D2'): 0
}

# 2. Construção do Modelo
model = pyo.ConcreteModel()
model.x = pyo.Var(custos.keys(), within=pyo.NonNegativeReals)

# Função Objetivo: Minimizar custos totais
model.obj = pyo.Objective(
    expr=sum(model.x[i, j] * custos[i, j] for (i, j) in custos.keys()),
    sense=pyo.minimize
)

# 3. Restrições de Transporte

def regra_oferta(model, i):
    # Filtra apenas os arcos que saem do nó 'i'
    arcos_saida = [dest for (orig, dest) in custos.keys() if orig == i]

    # Se não houver arcos saindo deste nó, pulamos a restrição
    if not arcos_saida:
        return pyo.Constraint.Skip

    return sum(model.x[i, dest] for dest in arcos_saida) == oferta_ajustada[i]

model.res_oferta = pyo.Constraint(nos, rule=regra_oferta)

def regra_demanda(model, j):
    # Filtra apenas os arcos que entram no nó 'j'
    arcos_entrada = [orig for (orig, dest) in custos.keys() if dest == j]

    # Se não houver arcos entrando neste nó, pulamos a restrição
    if not arcos_entrada:
        return pyo.Constraint.Skip

    return sum(model.x[orig, j] for orig in arcos_entrada) == demanda_ajustada[j]

model.res_demanda = pyo.Constraint(nos, rule=regra_demanda)


# 3. Solução com GLPK
opt = pyo.SolverFactory('glpk')
opt.solve(model)

# 4. Exibição dos Resultados Reais (Filtrando o Buffer)
print(f"Custo Total de Transporte: Z = {pyo.value(model.obj)}")
print("-" * 30)
for (i, j) in custos.keys():
    valor = pyo.value(model.x[i, j])
    if valor > 0 and i != j: # Exibe apenas fluxos reais entre nós diferentes
        print(f"Rota {i} -> {j}: Quantidade = {valor}")


Custo Total de Transporte: Z = 20700.0
------------------------------
Rota P1 -> T2: Quantidade = 1000.0
Rota P2 -> T1: Quantidade = 1200.0
Rota T1 -> D1: Quantidade = 800.0
Rota T1 -> D2: Quantidade = 400.0
Rota T2 -> D2: Quantidade = 1000.0
Rota D2 -> D3: Quantidade = 500.0
